<a href="https://colab.research.google.com/github/filipsajtlava/dspracticum2025-tismaci/blob/master/homeworks/hw6/Copy_of_Llama3_2_(1B_and_3B)_Conversational.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth your local device, follow [our guide](https://docs.unsloth.ai/get-started/install-and-update). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


In [ ]:
!git clone https://github.com/filipsajtlava/dspracticum2025-tismaci
%cd dspracticum2025-tismaci/

Cloning into 'dspracticum2025-tismaci'...
remote: Enumerating objects: 2121, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (25/25), done.
^C
[Errno 2] No such file or directory: 'dspracticum2025-tismaci/'
/content/dspracticum2025-tismaci


In [ ]:
file_path = "homeworks/hw4/textovy_dataset.txt"
with open(file_path, 'r') as f:
    textovy_dataset_content = f.read()

print("Content loaded successfully. First 200 characters:")
print(textovy_dataset_content[:200])

Content loaded successfully. First 200 characters:
nazov knihy ==================anonymous-authors_black-pullet.pdf
2
The Black Pullet Or The Hen With The Golden
Eggs
Before beginning the subject, and to acquaint my readers of this profound Science, w


In [ ]:
# New Installation Cell: Ensuring unsloth and trl are correctly installed.
# Removed %%capture temporarily to show installation output for debugging.

import os, re

print("Starting installation of Unsloth and dependencies...")

if "COLAB_" not in "".join(os.environ.keys()):
    print("Installing unsloth for non-Colab environment...")
    !pip install unsloth
else:
    print("Installing unsloth and dependencies for Colab environment...")
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers_version = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")

    print(f"Installing core libraries including xformers ({xformers_version})...")
    !pip install bitsandbytes accelerate peft triton cut_cross_entropy unsloth_zoo {xformers_version}

    print("Installing trl...")
    !pip install trl

    print("Installing unsloth...")
    !pip install unsloth

    print("Installing other data processing dependencies...")
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer

print("Installation process concluded.")

Starting installation of Unsloth and dependencies...
Installing unsloth and dependencies for Colab environment...
Installing core libraries including xformers (xformers==0.0.32.post2)...
Installing trl...
Installing unsloth...
Installing other data processing dependencies...
Installation process concluded.


In [ ]:
# New Cell for Model Loading (from original Znh96WvcstqF)

from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 2x faster
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # 4bit for 405b!
    "unsloth/Mistral-Small-Instruct-2409",     # Mistral 22b 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!

    "unsloth/Llama-3.2-1B-bnb-4bit",           # NEW! Llama 3.2 models
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",

    "unsloth/Llama-3.3-70B-Instruct-bnb-4bit" # NEW! Llama 3.3 70B!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-bnb-4bit", # Changed to base model
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

==((====))==  Unsloth 2025.11.2: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
# New Cell for LoRA Adapters (from original 6bZsfBuZDeCL)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Already have LoRA adapters! We shall skip this step.


In [ ]:
# New Data Preparation Cell for Continued Pretraining (Pure Text)

from unsloth.chat_templates import get_chat_template
from datasets import Dataset

# Ensure the tokenizer is loaded with the chat template for potential inference later,
# but this template is NOT applied to the training data itself for continued pretraining.
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1", # Use the appropriate chat template for your model
)

# Create a dataset where the 'text' field contains your entire pure text content.
# We assume `textovy_dataset_content` is already loaded into the environment.
dataset_raw = [{
    "text": textovy_dataset_content
}]
dataset = Dataset.from_list(dataset_raw)

print("Dataset created for continued pretraining. First 500 characters of the text:")
print(dataset[0]["text"][:500])

Dataset created for continued pretraining. First 500 characters of the text:
nazov knihy ==================anonymous-authors_black-pullet.pdf
2
The Black Pullet Or The Hen With The Golden
Eggs
Before beginning the subject, and to acquaint my readers of this profound Science, which
until the present day has been the object of research of the most constant and profound
meditations, I must unbosom myself how these marvelous secrets were communicated to me,
and the manner in which the Divine Providence allowed me to escape from the greatest
dangers and, so to speak, conducte


For continued pretraining with pure text, we **do not** use conversational data standardization or apply a formatting function that transforms conversational data. The `dataset` object directly contains your pure text in the `text` field, ready for the trainer.

In [ ]:
# New Trainer Setup Cell for Continued Pretraining

from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset, # Use the dataset created from your pure text
    dataset_text_field = "text", # This specifies that the 'text' field contains the training data
    max_seq_length = max_seq_length,
    packing = True, # Highly recommended for continued pretraining for efficiency
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 500, # Increased max_steps to give the model more time to learn
        learning_rate = 2e-5,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

num_proc must be <= 1. Reducing num_proc to 1 for dataset of size 1.


Unsloth: Tokenizing ["text"]:   0%|          | 0/1 [00:00<?, ? examples/s]

For continued pretraining on pure text, we **do NOT** use `train_on_responses_only`. This function is designed for masking parts of *conversational* data, and in pure text pretraining, the model learns from the entire text provided in the `text` field.

In [ ]:
# Verification: Raw text content from the first dataset entry.
# The SFTTrainer with packing=True will handle tokenization and label creation internally.

print("Raw text content from the first dataset entry (what the trainer sees):\n")
print(dataset[0]["text"][:1000]) # Display first 1000 characters for inspection

Raw text content from the first dataset entry (what the trainer sees):

nazov knihy ==================anonymous-authors_black-pullet.pdf
2
The Black Pullet Or The Hen With The Golden
Eggs
Before beginning the subject, and to acquaint my readers of this profound Science, which
until the present day has been the object of research of the most constant and profound
meditations, I must unbosom myself how these marvelous secrets were communicated to me,
and the manner in which the Divine Providence allowed me to escape from the greatest
dangers and, so to speak, conducted me by the Divine Hand, to prove that by Divine Will it is
sufficient to raise unto Himself the last of Beings or to precipate to naught those who are
clothed with all power on Earth. We all therefor come from God, God is everything, and
without God nothing can exist. Who more than I may penetrate the truth eternal and sacred.
I formed part of the expedition to Egypt, an officer in the army of the genius. I took part in
the

In [ ]:
# @title Show current memory stats before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved before training.")

GPU = Tesla T4. Max memory = 14.741 GB.
11.189 GB of memory reserved before training.


In [ ]:
print("Starting model training...")
trainer_stats = trainer.train()
print("Training completed.")

The model is already on multiple devices. Skipping the move to device specified in `args`.


Starting model training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1 | Num Epochs = 500 | Total steps = 500
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
1,2.538900
2,2.538900
3,2.538600
4,2.537700
5,2.535400
6,2.530300
7,2.522400
8,2.513200
9,2.503300
10,2.493100


Training completed.


In [ ]:
# @title Show final memory and time stats after training
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

1259.4367 seconds used for training.
20.99 minutes used for training.
Peak reserved memory = 11.189 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 75.904 %.
Peak reserved memory for training % of max memory = 0.0 %.


Masked labels inspection (as typically done for conversational finetuning) is **not applicable** for pure text continued pretraining. The trainer handles tokenization and label creation internally for language modeling, where the model learns to predict the next token based on the preceding ones throughout the entire text.

For continued pretraining with pure text, we **do not** use conversational data standardization or apply a formatting function that transforms conversational data. The `dataset` object directly contains your pure text in the `text` field, ready for the trainer.

In [ ]:
FastLanguageModel.for_inference(model) # Ensure inference mode is enabled

messages = [
    {"role": "user", "content": "Create a magical recipe for an 'Elixir of Clarity'.\n\n**Ingredients:**\n* 1 drop of morning dew from a forgotten forest\n* 3 petals from a moonflower\n* 1 pinch of stardust\n\n**Preparation Steps:**\n1. Collect dew at dawn.\n2. Gently press moonflower petals into a vial.\n3. Add stardust to activate.\n\nNow, create a magical recipe for an 'Elixir of Dreams'. List the mystical ingredients needed and the detailed preparation steps to brew it."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 256,
                   use_cache = True, temperature = 0.9, min_p = 0.6) # Slightly adjusted parameters

Create a magical recipe for an 'Elixir of Clarity'.

**Ingredients:**
* 1 drop of morning dew from a forgotten forest
* 3 petals from a moonflower
* 1 pinch of stardust

**Preparation Steps:**
1. Collect dew at dawn.
2. Gently press moonflower petals into a vial.
3. Add stardust to activate.

Now, create a magical recipe for an 'Elixir of Dreams'. List the mystical ingredients needed and the detailed preparation steps to brew it..”



<|end_of_text|>


In [ ]:
FastLanguageModel.for_inference(model) # Ensure inference mode is enabled

messages = [
    {"role": "user", "content": "Create a magical recipe for an 'Elixir of Clarity'.\n\n**Ingredients:**\n* 1 drop of morning dew from a forgotten forest\n* 3 petals from a moonflower\n* 1 pinch of stardust\n\n**Preparation Steps:**\n1. Collect dew at dawn.\n2. Gently press moonflower petals into a vial.\n3. Add stardust to activate.\n\nNow, create a magical recipe for an 'Elixir of Dreams'. List the mystical ingredients needed and the detailed preparation steps to brew it."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 256,
                   use_cache = True, temperature = 1.5, min_p = 0.1) # Slightly adjusted parameters

Mystical Ingredients and Preparation Steps for the 'Elixir of Dreams':

**Ingredients:**
1. A piece of the moon's white cloak
2. A crystal ball of star dust
3. A golden vial with a cap
4. A bottle of black oil

**Preparation Steps:**
1. Gather the ingredients.
2. Wash the cloak and remove any dirt.
3. Place the cloak on the crystal ball.
4. Pour the black oil onto the crystal ball and mix.
5. Add the gold cap to the bottle.
6. Shake the bottle vigorously until a seal appears.
7. Store the vial in a safe place.

Remember, the elixir must be crafted with great care and respect for the nature spirits and elements to be truly powerful and beneficial.

圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭圭


In [ ]:
FastLanguageModel.for_inference(model) # Ensure inference mode is enabled

messages = [
    {"role": "user", "content": "Create a magical recipe for an 'Elixir of Clarity'.\n\n**Ingredients:**\n* 1 drop of morning dew from a forgotten forest\n* 3 petals from a moonflower\n* 1 pinch of stardust\n\n**Preparation Steps:**\n1. Collect dew at dawn.\n2. Gently press moonflower petals into a vial.\n3. Add stardust to activate.\n\nNow, create a magical recipe for an 'Elixir of Dreams'. List the mystical ingredients needed and the detailed preparation steps to brew it."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 256,
                   use_cache = True, temperature = 1.5, min_p = 0.1) # Slightly adjusted parameters

**Elixir of Clarity Recipe**
>1 drop of morning dew from a forgotten forest
>3 petals from a moonflower
>1 pinch of stardust

**Preparation Steps**
1. Collect dew at dawn.
2. Gently press moonflower petals into a vial.
3. Add stardust to activate.

**Ingredients for Elixir of Dreams Recipe**
* 1 moonstone
* 1 piece of dark velvet
* 1 bottle of clear essence
* 1 bottle of sweet essence
* 1 piece of lavender

**Preparation Steps**
1. Place moonstone and dark velvet in a small container.
2. Pour clear and sweet essence into a cauldron.
3. Light the lavender and allow to smolder for a few moments.
4. Pour the essence into the cauldron and stir until well combined.
5. Place the moonstone and dark velvet into a clear vial and seal tightly.

 McConnell is an amazing chef, using his skills to create these magical recipes. He truly has a gift for bringing together these mystical ingredients and creating these powerful elixirs. His attention to detail in each step is what makes them so powerful.

In [ ]:
FastLanguageModel.for_inference(model) # Ensure inference mode is enabled

messages = [
    {"role": "user", "content": "Create a magical recipe for an 'Elixir of Dreams'. List the mystical ingredients needed and the detailed preparation steps to brew it."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 256,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

In this tutorial, we will learn how to create and run a simple Elixir application using the Ecto framework and SQLite.

I. Prerequisites

Elixir 2.5, Erlang 23, and Redis

II. Application and Database Configuration

To create and run our application, we must first create a new directory, `magical_recipe` and initialize our project using `mix`:

`cd /path/to/magical_recipe`
`mix new magical_recipe`

This will create the necessary directory structure for our application and generate an initial `mix` executable.

We can then set up a local development SQLite database and seed our tables with some initial data. For this, we need to create a new file called `config.exs` and add some configuration for our SQLite database and Ecto migrations.

In our new `config.exs` file, we will set up the SQLite database using `use Mix.Config`, and configure the database adapter and URL, and seed our `magical_recipes` table with the following recipes:

```elixir
use Mix.Config

config :ecto_repos, MagicalR

In [ ]:
# --- Compare with the base model to see the effect of pretraining ---
# Uncomment the following block to load the base model for comparison

if True:
    from unsloth import FastLanguageModel
    # Load the base model without any LoRA adapters
    base_model, base_tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/Llama-3.2-3B-bnb-4bit", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    base_tokenizer = get_chat_template(
        base_tokenizer,
        chat_template = "llama-3.1",
    )
    FastLanguageModel.for_inference(base_model) # Enable native 2x faster inference
else:
    base_model = model
    base_tokenizer = tokenizer

print("Model loaded for comparison (either base or finetuned depending on uncommenting)...")

==((====))==  Unsloth 2025.11.2: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Model loaded for comparison (either base or finetuned depending on uncommenting)...


In [ ]:
# Run the same inference prompt on the (potentially) base model
messages_for_comparison = [
    {"role": "user", "content": "Create a magical recipe for an 'Elixir of Clarity'.\n\n**Ingredients:**\n* 1 drop of morning dew from a forgotten forest\n* 3 petals from a moonflower\n* 1 pinch of stardust\n\n**Preparation Steps:**\n1. Collect dew at dawn.\n2. Gently press moonflower petals into a vial.\n3. Add stardust to activate.\n\nNow, create a magical recipe for an 'Elixir of Dreams'. List the mystical ingredients needed and the detailed preparation steps to brew it."},
]
inputs_for_comparison = base_tokenizer.apply_chat_template(
    messages_for_comparison,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer_comparison = TextStreamer(base_tokenizer, skip_prompt = True)
print("\n--- Output from Comparison Model ---")
_ = base_model.generate(input_ids = inputs_for_comparison, streamer = text_streamer_comparison, max_new_tokens = 256,
                   use_cache = True, temperature = 1.5, min_p = 0.1)
print("\n--- End of Comparison Output ---")

print("\nNow, compare this output to the previous one from your finetuned model.")


--- Output from Comparison Model ---
1. A mixture of 2 tablespoons of olive oil and 3 tablespoons of honey in a container with a tight lid.
2. Place the container in the fridge and refrigerate overnight.
3. Take out the mixture and place the container on the stove at medium temperature (not too hot, nor too cold). While gently stirring the mixture with a wooden spoon.
4. Once the mixture has started to boil, add one spoonful of hot water, stirring as you add it.
5. Repeat step #4 three times.
6. When done, take out the mixture, cover with a cloth, and let the mixture rest in the refrigerator overnight.

`%Explain the process`
`%What do each of these mean to us?`
`%The purpose of each ingredient`
`%Their effects`

**What are the advantages to knowing all this?**
* Better understanding of how each step works

`%How to improve a recipe?`

**When you've created your 'Elixir of Clarity', let's test it out!**
```
iex(2)> Elirium.clear
[...]
```
**Let's try your 'Elixir of Dreams'. What does

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2024.10.0 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
# New Cell for Model Loading (from original Znh96WvcstqF)

from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 2x faster
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # 4bit for 405b!
    "unsloth/Mistral-Small-Instruct-2409",     # Mistral 22b 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!

    "unsloth/Llama-3.2-1B-bnb-4bit",           # NEW! Llama 3.2 models
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",

    "unsloth/Llama-3.3-70B-Instruct-bnb-4bit" # NEW! Llama 3.3 70B!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit", # Changed to base model
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

==((====))==  Unsloth 2025.11.2: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
# New Cell for LoRA Adapters (from original 6bZsfBuZDeCL)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

In [ ]:
# New Trainer Setup Cell for Continued Pretraining

from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset, # Use the dataset created from your pure text
    dataset_text_field = "text", # This specifies that the 'text' field contains the training data
    max_seq_length = max_seq_length,
    packing = True, # Highly recommended for continued pretraining for efficiency
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 100, # Increased max_steps to give the model more time to learn
        learning_rate = 2e-5,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

num_proc must be <= 1. Reducing num_proc to 1 for dataset of size 1.


Unsloth: Tokenizing ["text"]:   0%|          | 0/1 [00:00<?, ? examples/s]

In [ ]:
print("Starting model training...")
trainer_stats = trainer.train()
print("Training completed.")

The model is already on multiple devices. Skipping the move to device specified in `args`.


Starting model training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1 | Num Epochs = 100 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
1,2.822400
2,2.822400
3,2.822300
4,2.820900
5,2.817700
6,2.811300
7,2.801300
8,2.790100
9,2.778600
10,2.766900


Training completed.


In [ ]:
FastLanguageModel.for_inference(model) # Ensure inference mode is enabled

messages = [
    {"role": "user", "content": "Create a magical recipe for an 'Elixir of Clarity'."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 256,
                   use_cache = True, temperature = 1.5, min_p = 0.1) # Slightly adjusted parameters

**The Elixir of Clarity: A Magical Recipe**

In the realm of mysticism, the Elixir of Clarity is a rare and precious potion said to grant the drinker clarity of mind, focus, and mental agility. This enchanted brew requires precision, patience, and a deep understanding of the universe's harmony.

**Ingredients:**

1. **Luminous Petal of Clarity**: A rare, delicate flower that blooms only under the light of a full moon. Its petals are said to hold the essence of clarity and focus.
2. **Golden Ambrosia**: A fragment of the golden nectar collected from the ambrosial tree of ancient wisdom. This sacred ambrosia amplifies the elixir's effects.
3. **Mindstone Essence**: A crystalline extract from the heart of the Mindstone, a mystical mountain said to harbor the secrets of the universe. This essence enhances cognitive functions and mental clarity.
4. **Stardust Dusting Powder**: A pinch of stardust collected from a shooting star's trail. This magical powder imbues the elixir with cosmic energ

In [ ]:
FastLanguageModel.for_inference(model) # Ensure inference mode is enabled

messages = [
    {"role": "user", "content": "Create a magical recipe for an 'Elixir of Clarity'."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 256,
                   use_cache = True, temperature = 0.9, min_p = 0.6) # Slightly adjusted parameters

**The Elixir of Clarity Recipe**

**Ingredients:**

1. **Moonpetal Blooms**: Rare, delicate flowers that only bloom under the light of a full moon. Their petals hold the essence of clarity and focus.
2. **Starlight Salt**: A pinch of salt harvested from the heart of a crystal cave, imbued with the twinkling magic of the stars.
3. **Luminous Leaves**: Leaves from the ancient Luminous Tree, which absorb and store the light of a thousand suns. Their essence is said to grant wisdom and insight.
4. **Crystal Clear Water**: A vial of water collected from the purest mountain spring, where the whispers of the forest are said to reside.
5. **Golden Honey**: A drizzle of pure, golden honey harvested from the hives of enchanted bees, known for their ability to sense the vibrations of the universe.

**Instructions:**

1. Under the light of a full moon, gently harvest the Moonpetal Blooms and place them in a silver bowl.
2. Add a pinch of Starlight Salt to the bowl, stirring clockwise to infuse the

# Without training

In [ ]:
# --- Compare with the base model to see the effect of pretraining ---
# Uncomment the following block to load the base model for comparison

if True:
    from unsloth import FastLanguageModel
    # Load the base model without any LoRA adapters
    base_model, base_tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    base_tokenizer = get_chat_template(
        base_tokenizer,
        chat_template = "llama-3.1",
    )
    FastLanguageModel.for_inference(base_model) # Enable native 2x faster inference
else:
    base_model = model
    base_tokenizer = tokenizer

print("Model loaded for comparison (either base or finetuned depending on uncommenting)...")

==((====))==  Unsloth 2025.11.2: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Model loaded for comparison (either base or finetuned depending on uncommenting)...


In [ ]:
# Run the same inference prompt on the (potentially) base model
messages_for_comparison = [
    {"role": "user", "content": "Create a magical recipe for an 'Elixir of Clarity'."},
]
inputs_for_comparison = base_tokenizer.apply_chat_template(
    messages_for_comparison,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer_comparison = TextStreamer(base_tokenizer, skip_prompt = True)
print("\n--- Output from Comparison Model ---")
_ = base_model.generate(input_ids = inputs_for_comparison, streamer = text_streamer_comparison, max_new_tokens = 256,
                   use_cache = True, temperature = 1.5, min_p = 0.1)
print("\n--- End of Comparison Output ---")

print("\nNow, compare this output to the previous one from your finetuned model.")


--- Output from Comparison Model ---
**Elixir of Clarity Recipe**

In a realm where magic and mystery entwine, we present to you a recipe for the enchanted Elixir of Clarity. This mystical drink is said to grant clarity of mind, sharpness of thought, and insight into the unknown.

**Ingredients:**

1. **Moonpetal Blooms**: Rare and exquisite, these delicate flowers bloom only once a year, under the light of a full moon. Harvest the petals at dawn, when the dew is still fresh.
2. **Starlight Salt**: A pinch of salt harvested from a celestial spring, imbued with the essence of stardust and the whispers of the cosmos.
3. **Dreamweaver's Honey**: A drizzle of pure, golden nectar collected from the hives of rare, moon-pollinated bees.
4. **Crisp Clearwater**: Freshwater collected from a crystal-clear lake, filtered through the leaves of the Luminous Fern.
5. **Essence of Luminous Moss**: A subtle, shimmering extract obtained from the delicate, glowing threads of the Luminous Moss that grow

In [ ]:
# New Trainer Setup Cell for Continued Pretraining

from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset, # Use the dataset created from your pure text
    dataset_text_field = "text", # This specifies that the 'text' field contains the training data
    max_seq_length = max_seq_length,
    packing = True, # Highly recommended for continued pretraining for efficiency
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 300, # Increased max_steps to give the model more time to learn
        learning_rate = 2e-5,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

num_proc must be <= 1. Reducing num_proc to 1 for dataset of size 1.


Unsloth: Tokenizing ["text"]:   0%|          | 0/1 [00:00<?, ? examples/s]

In [ ]:
print("Starting model training...")
trainer_stats = trainer.train()
print("Training completed.")

The model is already on multiple devices. Skipping the move to device specified in `args`.


Starting model training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1 | Num Epochs = 300 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
1,2.822400
2,2.822400
3,2.822300
4,2.820900
5,2.817700
6,2.811300
7,2.801300
8,2.790100
9,2.778400
10,2.766500


Training completed.


In [ ]:
FastLanguageModel.for_inference(model) # Ensure inference mode is enabled

messages = [
    {"role": "user", "content": "Create a magical recipe for an 'Elixir of Clarity'."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 256,
                   use_cache = True, temperature = 1.5, min_p = 0.1) # Slightly adjusted parameters

**The Elixir of Clarity: A Magical Recipe**

In the realm of mysticism, the Elixir of Clarity is a rare and precious potion said to grant the drinker clarity of mind, focus, and mental agility. This enchanted brew requires the finest ingredients and a pinch of magical intention. Gather the following ingredients and prepare to brew the Elixir of Clarity.

**Ingredients:**

1. **Moonpetal Blooms**: These delicate, silver-petaled flowers bloom only under the light of a full moon. Harvest their essence by steeping them in pure mountain spring water under the starry night sky.
2. **Starlight Salt**: Harvest salt from a sacred crystal cave, where the essence of stardust has infused the crystals. Grind the salt into a fine powder to enhance its magical properties.
3. **Golden Ambrosia Honey**: Collect the nectar of sunflowers and honey bees from a field where the sun always shines. This rare honey holds the power of pure intentions and optimism.
4. **Dragon's Blood Berries**: Forged in the he

In [ ]:
FastLanguageModel.for_inference(model) # Ensure inference mode is enabled

messages = [
    {"role": "user", "content": "Create a magical recipe for an 'Elixir of Clarity'."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 256,
                   use_cache = True, temperature = 0.9, min_p = 0.6) # Slightly adjusted parameters

**The Ancient Recipe for the Elixir of Clarity**

In the mystical realm of Aethoria, where the sun dipped into the horizon and painted the sky with hues of sapphire and amethyst, the wise sorceress, Lyra, revealed to us the secrets of the Elixir of Clarity. This enchanted potion is said to grant the drinker unparalleled mental acuity, clarity of thought, and unwavering focus.

**Ingredients:**

1. **Moonpetal Blooms**: Rare, delicate flowers that bloom only under the light of the full moon. Their petals hold the essence of lunar magic, which amplifies the drinker's intuition and psychic abilities.
2. **Starlight Dust**: A pinch of stardust collected from the heart of a shooting star. This celestial ingredient imbues the elixir with the power of celestial guidance and wisdom.
3. **Essence of Clarity**: A vial of pure, crystalline water distilled from the crystal caves of the ancient mountains. This essence clarifies the mind, allowing the drinker to see through deception and illusions.
